# 知识点结构相似度评估器

功能：评估一批试题样本的知识点结构与基准试题库的知识点结构的结构相似度。

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.graphs import Neo4jGraph
import re
import numpy as np
from collections import defaultdict
from langchain.chat_models import init_chat_model

In [2]:
# Neo4j 图数据库连接
os.environ["DEEPSEEK_API_KEY"] = "sk-ae6d423f13b7403fa8ba16ddb4acc297"
os.environ["QWEN_API_KEY"] = "sk-9b4b258c64544e609ff95dae918a401f"

url = "bolt://localhost:7689"
username = "neo4j"
password = "12345678"

load_dotenv()
graph = Neo4jGraph(
    url=url,
    username=username,
    password=password
)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_50208\698575476.py:10: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [18]:
# 定义知识点库（各小节的知识点列表）
knowledge_base = {
    '流水地貌': ["地貌", "流水地貌", "流水侵蚀地貌", "流水堆积地貌", "峡谷", "河漫滩", "河流阶地", "曲流", "牛轭湖", "冲积扇", "洪积扇", 
             "冲积平原", "三角洲", "江心洲", "滑坡", "泥石流", "流水", "山洪", "虎跳峡",
             "雅鲁藏布大峡谷", "嘉陵江", "崇明岛", "橘子洲", "黄河三角洲", "尼罗河三角洲", "华北平原", "东北平原", "长江中下游平原"],
    '风成地貌': ["风蚀地貌", "风积地貌", "风蚀蘑菇", "风蚀壁龛", "风蚀柱", "雅丹地貌", "沙丘", "新月形沙丘", "灌丛沙丘", "风力"],
    '喀斯特、海岸和冰川地貌': ["喀斯特地貌", "海岸地貌", "冰川地貌", "海蚀地貌", "海积地貌", "海蚀崖", "海蚀平台", "海蚀柱", "海滩", 
                    "岬角", "冰斗", "冰川槽谷", "角峰", "刃脊", "峡湾", "波浪", "冰川", "云南石林", "广西桂林", "重庆奉节小寨天坑", 
                    "贵州荔波", "四川黄龙", "湖南张家界黄龙洞", "澳大利亚坎贝尔港国家公园", "挪威西峡湾"]
}

# 展平为单层列表，便于匹配
all_knowledge_points = []
for section, points in knowledge_base.items():
    all_knowledge_points.extend(points)
print(f"知识点库共 {len(all_knowledge_points)} 个知识点")

知识点库共 63 个知识点


## 1. 从试题文本抽取知识点

In [7]:
# 初始化LLM
llm = init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=0.3
)

In [125]:
def extract_knowledge_points(question_text, options_text, knowledge_points_list):
    """
    从试题文本中抽取知识点（必须来自知识点库）
    使用LLM抽取，然后过滤只保留知识点库中的点
    """
    full_text = f"{question_text}\n{options_text}"
    print(full_text)
    
    # 构建知识点列表字符串供LLM参考
    kp_str = "、".join(knowledge_points_list[:20])  # 限制长度
    
    prompt = f"""
        你是一个地理知识抽取助手。判断试题文本中涉及了以下哪个/些知识点。
        
        候选知识点：{knowledge_points_list}
        
        试题文本：
        {full_text}
        
        要求：
        1. 只输出试题涉及的知识点，不要遗漏，用中文顿号分隔
        2. 如果没有知识点出现，输出"无"
        3. 只输出知识点名称，不要解释
        4. 数量上实事求是，可能是一个也可能是多个，客观判断
        """
    
    try:
        response = llm.invoke(prompt)
        content = response.content.strip() if hasattr(response, 'content') else str(response).strip()
        
        if content == "无" or not content:
            return []
        
        # 解析LLM输出，过滤出有效的知识点
        extracted = [kp.strip() for kp in content.split('、')]
        valid_kps = [kp for kp in extracted if kp in knowledge_points_list]
        return valid_kps
    except Exception as e:
        print(f"抽取出错: {e}")
        return []

# 测试
test_q = "在黄土高原地区，流水侵蚀作用强烈，形成了塬、梁、峁等独特的地貌形态。"
test_o = "A. 地表径流沿黄土垂直节理下切，形成深沟 B. 季节性洪水携带大量泥沙在沟口堆积 C. 风力搬运的沙粒对黄土坡面进行磨蚀 D. 冰川在黄土覆盖区刨蚀地面"
result = extract_knowledge_points(test_q, test_o, all_knowledge_points)
print(f"抽取结果: {result}")

在黄土高原地区，流水侵蚀作用强烈，形成了塬、梁、峁等独特的地貌形态。
A. 地表径流沿黄土垂直节理下切，形成深沟 B. 季节性洪水携带大量泥沙在沟口堆积 C. 风力搬运的沙粒对黄土坡面进行磨蚀 D. 冰川在黄土覆盖区刨蚀地面
抽取结果: ['流水侵蚀地貌', '流水地貌', '地貌']


In [126]:
# ========== 改进版：向量检索粗筛 + LLM精筛 ==========

# 预计算知识点embeddings（使用SentenceTransformer）
from sentence_transformers import SentenceTransformer, util
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 一次性计算所有知识点的embedding
knowledge_embeddings = embedder.encode(all_knowledge_points, convert_to_tensor=True)
print(f"已预计算 {len(all_knowledge_points)} 个知识点的embeddings")

def get_knowledge_candidates(question_text, options_text, top_k=12):
    """
    阶段1：向量检索粗筛
    用试题文本检索向量库，找出top-k最可能相关的知识点作为候选
    """
    full_text = f"{question_text}\n{options_text}"
    q_emb = embedder.encode(full_text, convert_to_tensor=True)
    
    # 计算余弦相似度
    scores = util.cos_sim(q_emb, knowledge_embeddings)[0].cpu().numpy()
    
    # 取top_k
    top_idx = np.argsort(scores)[-top_k:]
    candidates = [all_knowledge_points[i] for i in top_idx]
    
    return candidates

def extract_knowledge_points_v2(question_text, options_text, knowledge_points_list):
    """
    改进版知识点抽取：向量检索粗筛 + LLM精筛
    
    1. 先用向量检索找出top-12候选知识点
    2. 让LLM只在这12个候选中确认哪些出现在试题中
    """
    full_text = f"{question_text}\n{options_text}"
    
    # 阶段1：向量检索粗筛
    candidates = get_knowledge_candidates(question_text, options_text, top_k=12)
    
    # 阶段2：LLM精筛（只判断12个候选）
    candidates_str = "、".join(candidates)
    
    prompt = f"""
        你是一个地理知识抽取助手。判断试题文本中涉及了以下哪个/些知识点。
        
        候选知识点：{candidates_str}
        
        试题文本：
        {full_text}
        
        要求：
        1. 只输出试题涉及的知识点，不要遗漏，用中文顿号分隔
        2. 如果没有知识点出现，输出"无"
        3. 只输出知识点名称，不要解释
        4. 数量上实事求是，可能是一个也可能是多个，客观判断
        """

    try:
        response = llm.invoke(prompt)
        content = response.content.strip() if hasattr(response, 'content') else str(response).strip()
        
        if content == "无" or not content:
            return []
        
        # 解析LLM输出，过滤有效知识点
        extracted = [kp.strip() for kp in content.split('、')]
        valid_kps = [kp for kp in extracted if kp in knowledge_points_list]
        
        # 双重保险：确保抽取的知识点确实在候选列表中
        valid_kps = [kp for kp in valid_kps if kp in candidates]
        
        return valid_kps
    except Exception as e:
        print(f"抽取出错: {e}")
        return []

# 测试改进版
result = extract_knowledge_points_v2(test_q, test_o, all_knowledge_points)
print(f"改进版抽取结果: {result}")

# 可选：对比向量检索候选和最终结果
candidates = get_knowledge_candidates(test_q, test_o)
print(f"向量检索候选: {candidates}")

已预计算 63 个知识点的embeddings
改进版抽取结果: ['流水侵蚀地貌', '流水地貌']
向量检索候选: ['海岸地貌', '海蚀地貌', '海积地貌', '地貌', '河流阶地', '雅丹地貌', '风蚀地貌', '风积地貌', '喀斯特地貌', '流水地貌', '流水侵蚀地貌', '流水堆积地貌']


In [62]:
# 从题库构建临时知识图谱
import pandas as pd
def build_knowledge_graph_bank(txt_file_path, use_v2=True):
    """
    从试题文件构建知识点共现图
    输入: xlsx题库
    输出: (nodes_dict, edges_dict) - 节点权重字典和边权重字典
    use_v2: 是否使用改进版抽取（向量检索+LLM精筛）
    """
    # 节点权重：知识点出现次数
    node_weights = defaultdict(int)
    # 边权重：知识点共现次数
    edge_weights = defaultdict(int)

    df = pd.read_excel(txt_file_path, dtype=str)  # 全部按字符串读取
    bank_items = []  # 每个元素为 (节, 难度, 向量)

    for idx, row in df.iterrows():

        # 组合完整试题文本：材料 + 提问 + 选项1~4
        question_text=[
            str(row.iloc[2]) if pd.notna(row.iloc[2]) else '',  # 材料
            str(row.iloc[3]) if pd.notna(row.iloc[3]) else '',  # 提问
        ]
        options_text = [
            str(row.iloc[2]) if pd.notna(row.iloc[2]) else '',  # 材料
            str(row.iloc[3]) if pd.notna(row.iloc[3]) else '',  # 提问
            str(row.iloc[4]) if pd.notna(row.iloc[4]) else '',  # 选项1
            str(row.iloc[5]) if pd.notna(row.iloc[5]) else '',  # 选项2
            str(row.iloc[6]) if pd.notna(row.iloc[6]) else '',  # 选项3
            str(row.iloc[7]) if pd.notna(row.iloc[7]) else '',  # 选项4
        ]
        
        question_text = ' '.join([p for p in question_text if p.strip()])
        options_text = ' '.join([p for p in options_text if p.strip()])

        # 抽取知识点（使用改进版）
        if use_v2:
            kps = extract_knowledge_points_v2(question_text, options_text, all_knowledge_points)
        else:
            kps = extract_knowledge_points(question_text, options_text, all_knowledge_points)
        
        if not kps:
            print(f'第{idx+1}题未抽取出知识点')
            continue
        
        print(f"第{idx+1}题: {kps}")
        
        # 更新节点权重
        for kp in kps:
            node_weights[kp] += 1
        
        # 更新边权重（知识点两两共现）
        for j in range(len(kps)):
            for k in range(j+1, len(kps)):
                edge = tuple(sorted([kps[j], kps[k]]))
                edge_weights[edge] += 1
    
    return dict(node_weights), dict(edge_weights)

# 构建样本知识点图
tmp_path='./data/题库 _已标注.xlsx'
nodes0, edges0 = build_knowledge_graph_bank(tmp_path, use_v2=True)
# print(f"\n节点: {nodes}")
# print(f"边: {edges}")

第1题: ['流水', '长江中下游平原', '流水侵蚀地貌']
第2题: ['流水', '流水地貌', '流水侵蚀地貌']
第3题: ['流水', '曲流']
第4题: ['流水堆积地貌']
第5题: ['流水堆积地貌']
第6题: ['流水堆积地貌']
第7题: ['流水', '流水侵蚀地貌']
第8题: ['峡谷', '流水', '流水地貌', '流水侵蚀地貌']
第9题: ['流水堆积地貌']
第10题: ['流水', '流水堆积地貌']
第11题: ['冲积平原', '流水堆积地貌']
第12题: ['泥石流']
第13题: ['泥石流']
第14题: ['泥石流']
第15题: ['泥石流', '流水堆积地貌']
第16题: ['泥石流']
第17题: ['流水地貌', '流水堆积地貌', '流水侵蚀地貌']
第18题: ['流水侵蚀地貌', '流水堆积地貌']
第19题: ['冲积平原', '华北平原']
第20题: ['华北平原']
第21题: ['流水', '流水地貌']
第22题: ['流水地貌', '流水堆积地貌']
第23题: ['河漫滩', '流水堆积地貌']
第24题: ['流水', '流水堆积地貌']
第25题: ['泥石流']
第26题未抽取出知识点
第27题: ['流水侵蚀地貌', '曲流']
第28题: ['冲积平原', '曲流']
第29题: ['地貌', '风蚀地貌', '雅丹地貌']
第30题: ['雅丹地貌', '风蚀地貌']
第31题: ['长江中下游平原']
第32题: ['风蚀地貌']
第33题: ['风蚀地貌', '风积地貌']
第34题: ['新月形沙丘']
第35题: ['广西桂林']
第36题未抽取出知识点
第37题: ['风力', '流水', '流水堆积地貌', '流水侵蚀地貌']
第38题: ['流水', '流水地貌', '流水堆积地貌']
第39题: ['风积地貌', '风力']
第40题: ['新月形沙丘']
第41题: ['新月形沙丘']
第42题未抽取出知识点
第43题: ['冰川地貌', '云南石林']
第44题: ['流水', '流水地貌', '流水侵蚀地貌', '流水堆积地貌']
第45题: ['海蚀地貌', '海岸地貌', '喀斯特地貌', '流水侵蚀地貌']
第46题: ['海积地貌',

In [144]:
def build_knowledge_graph(txt_file_path, use_v2=True):
    """
    从试题文件构建知识点共现图
    输入: txt文件路径，字段分号分隔，第6字段题干，第7字段选项
    输出: (nodes_dict, edges_dict) - 节点权重字典和边权重字典
    use_v2: 是否使用改进版抽取（向量检索+LLM精筛）
    """
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    # 节点权重：知识点出现次数
    node_weights = defaultdict(int)
    # 边权重：知识点共现次数
    edge_weights = defaultdict(int)
    
    for i, line in enumerate(lines):
        parts = line.split(';')
        if len(parts) < 7:
            print(i+1,'字段不足')
            continue
        
        question_text = parts[5]
        options_text = parts[6]
        
        # 抽取知识点（使用改进版）
        if use_v2:
            kps = extract_knowledge_points_v2(question_text, options_text, all_knowledge_points)
        else:
            kps = extract_knowledge_points(question_text, options_text, all_knowledge_points)
        
        if not kps:
            print(f'第{i+1}题未抽取出知识点')
            continue
        
        print(f"第{i+1}题: {kps}")
        
        # 更新节点权重
        for kp in kps:
            node_weights[kp] += 1
        
        # 更新边权重（知识点两两共现）
        for j in range(len(kps)):
            for k in range(j+1, len(kps)):
                edge = tuple(sorted([kps[j], kps[k]]))
                edge_weights[edge] += 1
    
    return dict(node_weights), dict(edge_weights)

# 构建样本知识点图
tmp_path='./output/comparison/structure_similarity/qwen-graph-rag.txt'
nodes11, edges11 = build_knowledge_graph(tmp_path, use_v2=True)
print('已完成')
# print(f"\n节点: {nodes}")
# print(f"边: {edges}")

第1题: ['流水', '流水侵蚀地貌']
第2题: ['峡谷', '流水侵蚀地貌', '流水']
第3题: ['流水地貌', '流水堆积地貌', '长江中下游平原']
第4题: ['尼罗河三角洲', '三角洲']
第5题: ['流水堆积地貌', '流水']
第6题: ['流水侵蚀地貌']
第7题: ['流水侵蚀地貌', '泥石流']
第8题: ['流水侵蚀地貌']
第9题: ['河流阶地', '流水侵蚀地貌', '流水堆积地貌']
第10题: ['流水', '流水地貌', '曲流']
第11题: ['流水地貌', '流水堆积地貌']
第12题: ['河漫滩']
第13题: ['流水堆积地貌', '流水地貌', '流水']
第14题: ['曲流']
第15题: ['雅鲁藏布大峡谷', '峡谷', '流水侵蚀地貌']
第16题: ['流水侵蚀地貌', '泥石流']
第17题: ['流水', '峡谷', '流水地貌', '流水堆积地貌', '流水侵蚀地貌']
第18题: ['华北平原', '流水堆积地貌']
第19题: ['流水堆积地貌', '流水地貌']
第20题: ['长江中下游平原', '流水地貌', '流水堆积地貌']
第21题: ['峡谷', '流水侵蚀地貌']
第22题: ['流水堆积地貌', '流水']
第23题: ['雅鲁藏布大峡谷', '流水侵蚀地貌']
第24题: ['东北平原', '流水堆积地貌']
第25题: ['山洪']
第26题: ['流水', '流水侵蚀地貌', '流水堆积地貌', '曲流']
第27题: ['长江中下游平原', '流水地貌', '流水堆积地貌']
第28题: ['流水地貌', '流水堆积地貌']
第29题: ['峡谷', '流水侵蚀地貌']
第30题: ['流水', '流水地貌', '流水堆积地貌', '流水侵蚀地貌']
第31题: ['河漫滩', '流水', '流水地貌', '流水堆积地貌']
第32题: ['橘子洲', '流水地貌', '流水堆积地貌']
第33题: ['流水地貌', '流水堆积地貌']
第34题: ['流水', '流水地貌']
第35题: ['流水', '流水堆积地貌']
第36题: ['长江中下游平原', '流水地貌', '流水堆积地貌']
第37题: ['流水', '流水地貌', '流水堆积地貌'

In [145]:
print(len(nodes11))

40


## 3. 从Neo4j获取基准知识点图

In [29]:
def get_neo4j_knowledge_graph(section=None):
    """
    从Neo4j获取知识点图结构
    返回: (nodes_dict, edges_dict)
    """
    # 获取所有节点及其权重（出现次数）
    nodes_query = """
    MATCH (n)
    RETURN n.id as id
    """
    nodes_result = graph.query(nodes_query)
    node_weights = {}
    for record in nodes_result:
        node_id = record['id']
        # 这里简化处理，假设每个节点初始权重为1
        # 实际可以从Neo4j属性中获取更精确的权重
        node_weights[node_id] = 1
    
    # 获取所有边及其权重
    edges_query = """
    MATCH (a)-[r]-(b)
    RETURN a.id as source, b.id as target
    """
    edges_result = graph.query(edges_query)
    edge_weights = {}
    for record in edges_result:
        source = record['source']
        target = record['target']
        edge = tuple(sorted([source, target]))
        edge_weights[edge] = edge_weights.get(edge, 0) + 1
    
    return node_weights, edge_weights

# 获取基准图
neo4j_nodes, neo4j_edges = get_neo4j_knowledge_graph()
print(f"Neo4j节点数: {len(neo4j_nodes)}")
print(f"Neo4j边数: {len(neo4j_edges)}")

Neo4j节点数: 69
Neo4j边数: 67


## 4. 计算加权邻接矩阵相似度

In [146]:
def compute_weighted_adjacency_similarity(nodes1, edges1, nodes2, edges2):
    """
    计算两个知识图结构的加权邻接矩阵相似度
    
    方法：
    1. 找出两个图的公共节点
    2. 构建公共节点对应的加权邻接矩阵
    3. 使用余弦相似度计算矩阵相似度
    """
    # 找出公共节点
    common_nodes = set(nodes1.keys()) & set(nodes2.keys())
    if not common_nodes:
        return 0.0
    
    common_nodes = list(common_nodes)
    n = len(common_nodes)
    node_to_idx = {node: i for i, node in enumerate(common_nodes)}
    
    # 构建加权邻接矩阵
    matrix1 = np.zeros((n, n))
    matrix2 = np.zeros((n, n))
    
    # 填充矩阵1（样本图）
    for edge, weight in edges1.items():
        u, v = edge
        if u in node_to_idx and v in node_to_idx:
            i, j = node_to_idx[u], node_to_idx[v]
            matrix1[i, j] = weight
            matrix1[j, i] = weight
    
    # 填充矩阵2（基准图）
    for edge, weight in edges2.items():
        u, v = edge
        if u in node_to_idx and v in node_to_idx:
            i, j = node_to_idx[u], node_to_idx[v]
            matrix2[i, j] = weight
            matrix2[j, i] = weight
            
    # 归一化 重要！消除了两批题目容量差距带来的相似度损失
    matrix1 /= matrix1.sum()
    matrix2 /= matrix2.sum()
    
    # 计算余弦相似度
    flat1 = matrix1.flatten()
    flat2 = matrix2.flatten()
    
    norm1 = np.linalg.norm(flat1)
    norm2 = np.linalg.norm(flat2)
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
    
    similarity = np.dot(flat1, flat2) / (norm1 * norm2)
    
    return float(similarity)

# 测试相似度计算
sim = compute_weighted_adjacency_similarity(nodes11, edges11, nodes0, edges0)
print(f"加权邻接矩阵相似度: {sim:.4f}")

加权邻接矩阵相似度: 0.7797


In [81]:
print(nodes)
print(edges)
print(nodes2)
print(edges2)
print(nodes3)
print(edges3)

{'流水侵蚀地貌': 30, '流水': 13, '东北平原': 3, '泥石流': 4, '流水堆积地貌': 43, '江心洲': 9, '海积地貌': 1, '海岸地貌': 1, '华北平原': 7, '雅鲁藏布大峡谷': 7, '峡谷': 12, '曲流': 7, '流水地貌': 18, '长江中下游平原': 15, '嘉陵江': 3, '雅丹地貌': 2, '地貌': 4, '河流阶地': 7, '尼罗河三角洲': 2, '三角洲': 3, '黄河三角洲': 1, '山洪': 2, '冲积平原': 5, '喀斯特地貌': 1, '风蚀地貌': 1, '河漫滩': 1, '云南石林': 1}
{('流水', '流水侵蚀地貌'): 7, ('江心洲', '流水'): 4, ('江心洲', '流水堆积地貌'): 9, ('流水', '流水堆积地貌'): 8, ('海岸地貌', '海积地貌'): 1, ('流水侵蚀地貌', '雅鲁藏布大峡谷'): 5, ('峡谷', '雅鲁藏布大峡谷'): 5, ('流水', '雅鲁藏布大峡谷'): 1, ('峡谷', '流水侵蚀地貌'): 8, ('峡谷', '流水'): 2, ('曲流', '流水侵蚀地貌'): 6, ('江心洲', '流水地貌'): 3, ('流水地貌', '流水堆积地貌'): 15, ('华北平原', '流水堆积地貌'): 2, ('流水堆积地貌', '长江中下游平原'): 11, ('嘉陵江', '曲流'): 1, ('嘉陵江', '流水侵蚀地貌'): 2, ('流水地貌', '长江中下游平原'): 1, ('江心洲', '长江中下游平原'): 3, ('流水', '长江中下游平原'): 3, ('流水侵蚀地貌', '长江中下游平原'): 3, ('江心洲', '流水侵蚀地貌'): 2, ('流水侵蚀地貌', '流水堆积地貌'): 7, ('河流阶地', '流水堆积地貌'): 3, ('嘉陵江', '长江中下游平原'): 1, ('三角洲', '尼罗河三角洲'): 1, ('流水堆积地貌', '黄河三角洲'): 1, ('流水侵蚀地貌', '流水地貌'): 4, ('曲流', '长江中下游平原'): 3, ('河流阶地', '流水'): 1, ('河流阶地', '流水侵蚀地貌'): 2, ('曲流', '流

In [ ]:
def evaluate_structural_similarity(sample_file, section=None, use_v2=True):
    """
    评估样本知识点结构与基准图结构的相似度
    
    输入:
        sample_file: 样本文件路径
        section: 小节名称（可选，用于过滤）
        use_v2: 是否使用改进版抽取（向量检索+LLM精筛）
    输出:
        打印评估报告
    """
    print(f"正在评估文件: {sample_file}")
    
    # 1. 构建样本知识点图
    print("\n[1] 构建样本知识点共现图...")
    sample_nodes, sample_edges = build_knowledge_graph(sample_file, use_v2=use_v2)
    print(f"样本节点数: {len(sample_nodes)}")
    print(f"样本边数: {len(sample_edges)}")
    
    # 2. 获取Neo4j基准图
    print("\n[2] 获取Neo4j基准知识点图...")
    neo4j_nodes, neo4j_edges = get_neo4j_knowledge_graph(section)
    print(f"基准节点数: {len(neo4j_nodes)}")
    print(f"基准边数: {len(neo4j_edges)}")
    
    # 3. 计算相似度
    print("\n[3] 计算加权邻接矩阵相似度...")
    similarity = compute_weighted_adjacency_similarity(
        sample_nodes, sample_edges, 
        neo4j_nodes, neo4j_edges
    )
    
    print("\n" + "="*60)
    print(f"结构相似度得分: {similarity:.4f}")
    print("="*60)
    
    return similarity

In [ ]:
def evaluate_structural_similarity(sample_file, section=None):
    """
    评估样本知识点结构与基准图结构的相似度
    
    输入:
        sample_file: 样本文件路径
        section: 小节名称（可选，用于过滤）
    输出:
        打印评估报告
    """
    print(f"正在评估文件: {sample_file}")
    
    # 1. 构建样本知识点图
    print("\n[1] 构建样本知识点共现图...")
    sample_nodes, sample_edges = build_knowledge_graph(sample_file)
    print(f"样本节点数: {len(sample_nodes)}")
    print(f"样本边数: {len(sample_edges)}")
    
    # 2. 获取Neo4j基准图
    print("\n[2] 获取Neo4j基准知识点图...")
    neo4j_nodes, neo4j_edges = get_neo4j_knowledge_graph(section)
    print(f"基准节点数: {len(neo4j_nodes)}")
    print(f"基准边数: {len(neo4j_edges)}")
    
    # 3. 计算相似度
    print("\n[3] 计算加权邻接矩阵相似度...")
    similarity = compute_weighted_adjacency_similarity(
        sample_nodes, sample_edges, 
        neo4j_nodes, neo4j_edges
    )
    
    print("\n" + "="*60)
    print(f"结构相似度得分: {similarity:.4f}")
    print("="*60)
    
    return similarity

## 6. 批量评估多个文件

In [ ]:
import glob

def batch_evaluate_structural_similarity(folder_path, pattern="*.txt"):
    """
    批量评估文件夹中所有样本文件
    """
    files = glob.glob(os.path.join(folder_path, pattern))
    
    print(f"找到 {len(files)} 个文件\n")
    
    results = []
    
    for f in sorted(files):
        try:
            sim = evaluate_structural_similarity(f)
            results.append({
                'file': os.path.basename(f),
                'similarity': sim
            })
        except Exception as e:
            print(f"文件 {f} 评估失败: {e}")
    
    # 输出汇总
    print("\n" + "="*80)
    print("批量评估结果汇总")
    print("="*80)
    print(f"{'文件名':<50} {'结构相似度':>10}")
    print("-"*80)
    for r in results:
        print(f"{r['file']:<50} {r['similarity']:>10.4f}")
    
    avg_sim = np.mean([r['similarity'] for r in results])
    print("-"*80)
    print(f"{'平均相似度':<50} {avg_sim:>10.4f}")
    
    return results

# 测试批量评估
# batch_evaluate_structural_similarity("./output/comparison/raw", pattern="*noncontext*.txt")

# TOOL


In [ ]:
# 合并csv数据
from file_utils import concat_txt_files, concat_txt_files_with_counts

concat_txt_files_with_counts(
    ["./output/comparison/raw/qwen-210-noncontext_rel_ans_dif.txt",
     "./output/comparison/raw/qwen-220-noncontext_rel_ans_dif.txt",
     "./output/comparison/raw/qwen-230-noncontext_rel_ans_dif.txt",],
    './output/comparison/组合数据/qwen无检索3指标.txt',
    has_header=False)

各文件行数：
  ./output/comparison/raw/qwen-210-noncontext_rel_ans_dif.txt: 100 行
  ./output/comparison/raw/qwen-220-noncontext_rel_ans_dif.txt: 100 行
  ./output/comparison/raw/qwen-230-noncontext_rel_ans_dif.txt: 100 行
拼接完成：共 300 行，保存至 ./output/comparison/组合数据/qwen无检索3指标.txt


In [36]:
# 合并图
from collections import defaultdict

def merge_graphs(graph_list):
    """
    graph_list: [(nodes1, edges1), (nodes2, edges2), ...]
    返回合并后的 (merged_nodes, merged_edges)
    """
    merged_nodes = defaultdict(int)
    merged_edges = defaultdict(int)

    for nodes, edges in graph_list:
        # 合并节点
        for kp, w in nodes.items():
            merged_nodes[kp] += w
        
        # 合并边
        for (kp1, kp2), w in edges.items():
            edge = tuple(sorted([kp1, kp2]))
            merged_edges[edge] += w
    
    return dict(merged_nodes), dict(merged_edges)